# Analysis Notebook

## Some formal Analysis of the TN-TP tradeoff

* Select 6 fingerprinted models (1024, 4096, 8192 with/without instruct), 6 non-fp models
* Fix a couple of sampling strategies - 
    1. Greedy (accessed through `fp_dict['correct']`)
    2. Worst case adv Temperature (accessed through `max(fp_dict['mc_correct_detailed'].values())/10`)
    3. In top-k (accessed through `fp_dict['response_token_idx'] < k`)
* Fix a few values for $M$, num trials $T
* For $T$ trials-
    * Select $M$ fingerprints
    * Check number of positives across models for each FP
    * Create dict of 'model_name' to num_positives
    * For different values of $m$ (from 1 to $M$) label model as positive if num_positives >= m
    * Plot out val of true positives v/s true negatives (true positive is if fingerprinted model is tagged as positive, true negative if non-fingerprinted model is tagged as negative)

In [260]:
import random
import numpy as np
import matplotlib.pyplot as plt
import json

# =============================================================================
# 1. Model Selection
# =============================================================================
import re

def extract_params(string):
    if string == "greedy" or "ensemble" in string:
        return {"temperature": None, "top_p": None, "top_k": None, "min_p": None}
    
    pattern = r"temp_(?P<temperature>[\d\.]+)-p_(?P<top_p>[\d\.]+)-k_(?P<top_k>\d+)(?:-min_p_(?P<min_p>[\d\.]+))?"
    match = re.match(pattern, string)
    
    if match:
        params = match.groupdict()
        return {
            "temperature": float(params["temperature"]),
            "top_p": float(params["top_p"]),
            "top_k": int(params["top_k"]),
            "min_p": float(params["min_p"]) if params["min_p"] else None
        }
    return None


# Fingerprinted models (6 models)
fp_model_names = [
    "85d7f805f53c2c51e42e2780c668b045",
    "85d7f805f53c2c51e42e2780c668b045/ft_models/5461d82339fc7fb93d6fe167cce60b3c",
    "a795cae7c5974453e914ced9d2c736d1",
    "a795cae7c5974453e914ced9d2c736d1/ft_models/0a2d16228b00311de6dc8e8cf7bf1a18",
    "89238d0a3624e5d492066bc1d0153541",
    "89238d0a3624e5d492066bc1d0153541/ft_models/faef2f3c167b3233a800cf6b4128b307",
]
# Non-fingerprinted models (6 models)
non_fp_model_names = [
    "tokyotech-llm/Llama-3.1-Swallow-8B-v0.1",
    "google/gemma-2-9b",
    "google/gemma-2-9b-it",
    "MLP-KTLim/llama-3-Korean-Bllossom-8B",
    "DevsDoCode/LLama-3-8b-Uncensored",
    "teknium/OpenHermes-2.5-Mistral-7B",
    "allenai/Llama-3.1-Tulu-3-8B-DPO",
    "mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated",
    "NousResearch/Hermes-3-Llama-3.1-8B",
    "TheDrummer/Gemmasutra-9B-v1",
    "mistralai/Mistral-7B-v0.3",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "allenai/Llama-3.1-Tulu-3-8B-SFT",
    "ContactDoctor/Bio-Medical-Llama-3-8B"        
]

fp_file = "generated_data/output_fingerprints-inverse-nucleus-meta-llama-Meta-Llama-3.1-8B-response_length-16.json"

all_model_names = fp_model_names + non_fp_model_names

model_fp_results = {}

all_strats = set()

for model_name in all_model_names:
    benign_results_file = f"results/false_positive_analysis/all_fp/{fp_file.split('/')[-1].replace('.json', '')}-{model_name.replace('/', '-')}.json"
    adv_results_file = f"results/false_positive_analysis/all_fp_adversarial/{fp_file.split('/')[-1].replace('.json', '')}-{model_name.replace('/', '-')}.json"
    with open(benign_results_file, 'r') as f:
        benign_results = json.load(f)
    with open(adv_results_file, 'r') as f:
        adv_results = json.load(f)
    
    model_data = []
    
    for fp1, fp2 in zip(benign_results['data'], adv_results['data']):
        new_rec = {k:v for k,v in fp1.items()}
        for adv_keys in fp2['mc_correct_detailed'].keys():
            new_rec['mc_correct_detailed'][adv_keys] = fp2['mc_correct_detailed'][adv_keys]
            all_strats.add(adv_keys)
        for bk in fp1['mc_correct_detailed'].keys():
            new_rec['mc_correct_detailed'][bk] = fp1['mc_correct_detailed'][bk]
            all_strats.add(bk)
        model_data.append(new_rec)
    model_fp_results[model_name] = model_data


    

# =============================================================================
# 2. Define Strategy Functions
# =============================================================================
def strategy_greedy(fp_dict):
    # Greedy: simply use fp_dict['correct'] (assumed to be 0 or 1)
    return 1 if fp_dict['correct'] else 0

def strategy_min_adv(fp_dict, threshold=0.5):
    # Worst-case adversarial: take the max value from mc_correct_detailed,
    # normalize (assuming num_mc_trials=10), and compare to threshold.
    score = min(fp_dict['mc_correct_detailed'].values()) / 10.0
    return 1 if score >= threshold else 0

def strategy_max_adv(fp_dict, threshold=0.5):
    # Worst-case adversarial: take the max value from mc_correct_detailed,
    # normalize (assuming num_mc_trials=10), and compare to threshold.
    score = max(fp_dict['mc_correct_detailed'].values()) / 10.0
    return 1 if score >= threshold else 0

def strategy_in_top_k(fp_dict, k=10):
    # Positive if response token index is less than k.
    return 1 if fp_dict['response_token_idx'] < k else 0

def strategy_equals_k(fp_dict, k=0):
    return 1 if fp_dict['response_token_idx'] == k else 0

def strategy_in_k_but_not_top(fp_dict, k=10):
    if fp_dict['response_token_idx'] == 0:
        return 0
    return 1 if fp_dict['response_token_idx'] < k else 0
    # return 1 if  < k else 0

# Strategy to read off the mc_correct_detailed values
def strategy_mc_correct(fp_dict, mc_key, threshold=0.5, binary=True, do_sample=True):
    # Take the average of the mc_correct_detailed values

    if mc_key in fp_dict['mc_correct_detailed']:        
        score = fp_dict['mc_correct_detailed'][mc_key] / 10.0
    else:
        try:
            strat_params = extract_params(mc_key)
        except:
            strat_params = None
        if strat_params:
            if (mc_key + "-min_p_0.0") in fp_dict['mc_correct_detailed']:
                score = fp_dict['mc_correct_detailed'][mc_key + "-min_p_0.0"] / 10.0
                    # print(f"Key {mc_key} not found")
            else:
                score = fp_dict['mc_correct_detailed'][mc_key] / 10.0
                # raise ValueError(f"Key {mc_key} not found")
    # print(score)
    if binary:
        if not do_sample:
            return 1 if score >= threshold else 0
        else:
            return 1 if random.random() < score else 0
    else:
        return score

def strategy_ensemble_all(fp_dict, threshold=0.5):
    # Take the average of the mc_correct_detailed values
    score = sum(fp_dict['mc_correct_detailed'].values()) / (len(fp_dict['mc_correct_detailed']) * 10.0)
    # print(score)
    return 1 if score >= threshold else 0

# =============================================================================
# 3. Define Strategy Configurations (multiple parameter settings per strategy)
# =============================================================================
# Each entry: (strategy_name, strategy_function, parameter_dict)
strategy_configs = [
    ("greedy", strategy_greedy, {}),

    # Worst-case adversarial strategy with different thresholds.
    # ("worst_adv_thr_0.5", strategy_worst_adv, {"threshold": 0.5}),
    # ("worst_adv_thr_0.6", strategy_worst_adv, {"threshold": 0.6}),
    # ("worst_adv_thr_0.7", strategy_worst_adv, {"threshold": 0.7}),
    
    # ("max_adv_thr_0.1", strategy_max_adv, {"threshold": 0.1}),
    # ("max_adv_thr_0.2", strategy_max_adv, {"threshold": 0.2}),
    
    # ("min_adv_thr_0.9", strategy_min_adv, {"threshold": 0.9}),
    # ("min_adv_thr_0.8", strategy_min_adv, {"threshold": 0.8}),
    
    # # ensemble strategy
    ("ensemble_all_0.5", strategy_ensemble_all, {"threshold": 0.5}),
    ("ensemble_all_0.3", strategy_ensemble_all, {"threshold": 0.3}),
    ("ensemble_all_0.7", strategy_ensemble_all, {"threshold": 0.7}),
    # ("ensemble_all_0.1", strategy_ensemble_all, {"threshold": 0.1}),
    # ("ensemble_all_0.9", strategy_ensemble_all, {"threshold": 0.9}),
    # ("min_adv_thr_0.1", strategy_min_adv, {"threshold": 0.1}),
    # ("min_adv_thr_0.01", strategy_min_adv, {"threshold": 0.01}),
    # ("worst_adv_thr_0.5", strategy_worst_adv, {"threshold": 0.5}),    

    # In top-k strategy with different k values.
    # ("in_top_k_2", strategy_in_top_k, {"k": 2}),
    # ("in_top_k_5", strategy_in_top_k, {"k": 5}),
    # ("in_top_k_10", strategy_in_top_k, {"k": 10}),
    # ("in_top_k_15", strategy_in_top_k, {"k": 15}),
    # ("in_top_k_20", strategy_in_top_k, {"k": 20}),
    # ("in_top_k_50", strategy_in_top_k, {"k": 50}),
    # ("in_top_k_100", strategy_in_top_k, {"k": 100}),    
    
    # ("equals_k_1", strategy_equals_k, {"k": 1}),
    # ("equals_k_2", strategy_equals_k, {"k": 2}),
    # ("equals_k_5", strategy_equals_k, {"k": 5}),
    # ("equals_k_10", strategy_equals_k, {"k": 10}),
    
    # ("in_5_but_not_top_1", strategy_in_k_but_not_top, {"k": 5}),
    # ("in_10_but_not_top_2", strategy_in_k_but_not_top, {"k": 10}),
    # ("in_20_but_not_top_5", strategy_in_k_but_not_top, {"k": 20}),
    # ("in_k_but_not_top_10", strategy_in_k_but_not_top, {"k": 10}),
    
]

for strat in all_strats:
    # Canonicalize strategy name
    if strat.endswith('-min_p_0.0'):      
        # Remove the substring min_p-0.0 
        strategy_configs.append((strat.removesuffix('-min_p_0.0'), strategy_mc_correct, {"mc_key": strat.removesuffix('-min_p_0.0'), "binary": True}))
    else:
        strategy_configs.append((strat, strategy_mc_correct, {"mc_key": strat, "binary": True}))
    # strategy_configs.append((strat + "_score", strategy_mc_correct, {"mc_key": strat, "binary": False}))
from pprint import pprint
# pprint(strategy_configs)

# =============================================================================
# 4. Experimental Parameters
# =============================================================================
# List of M values (number of fingerprints to sample per trial per model)
M_values = [1, 3, 5, 10]
T = 1000  # Number of trials

# =============================================================================
# 5. Prepare Structures for Saving Results and for Plotting
# =============================================================================
# all_avg_roc will store the averaged ROC results for each M value.
all_avg_roc = {}  # keys: M, values: avg_roc dict for that M

# Create a figure with 2 rows (first row: TP vs TN; second row: TP vs FP)
# and one column per M value.
# fig, axes = plt.subplots(1, len(M_values), figsize=(6 * len(M_values), 5))

# =============================================================================
# 6. Main Analysis Loop for Different M Values
# =============================================================================
for i, M in enumerate(M_values):
    fig, ax = plt.subplots(figsize=FIG_SIZE_BIG)  # Main figure
    
    # Initialize the ROC data structure for current M.
    # roc_data[strategy_name][m] holds lists for TP, TN, FP, FN over T trials.
    roc_data = {
        strat_name: {m: {"TP": [], "TN": [], "FP": [], "FN": []} for m in range(0, M+2)}
        for strat_name, _, _ in strategy_configs
    }

    # Run T trials.
    for t in range(T):
        model_scores = {}  # model_name -> { strategy_name: score }
        for model_name in all_model_names:
            fingerprints = model_fp_results[model_name]
            # Sample M fingerprints (with replacement if necessary)
            if len(fingerprints) < M:
                selected = random.choices(fingerprints, k=M)
            else:
                selected = random.sample(fingerprints, M)

            scores = {}
            for strat_name, strat_fn, params in strategy_configs:
                # Sum the binary outcomes for each fingerprint
                try:
                    score = sum(strat_fn(fp, **params) for fp in selected)
                except:
                    score = 0
                    print(f"Error in {model_name} {strat_name}")
                scores[strat_name] = score
                # if strat_name in ["temp_0.9-p_0.9-k_50", "temp_0.9-p_0.9-k_50-min_p_0.0"] :
                #     print(model_name, strat_name, score)
            model_scores[model_name] = scores

        # For each threshold m = 1, ..., M, classify models and update counts.
        for m in range(0, M+2):
            for strat_name, _, _ in strategy_configs:
                tp = tn = fp_count = fn = 0
                for model_name, scores in model_scores.items():
                    predicted_positive = (scores[strat_name] >= m)
                    if model_name in fp_model_names:
                        if predicted_positive:
                            tp += 1
                        else:
                            fn += 1
                    else:
                        if not predicted_positive:
                            tn += 1
                        else:
                            fp_count += 1
                roc_data[strat_name][m]["TP"].append(tp)
                roc_data[strat_name][m]["TN"].append(tn)
                roc_data[strat_name][m]["FP"].append(fp_count)
                roc_data[strat_name][m]["FN"].append(fn)

    # Average the counts over T trials for each threshold.
    avg_roc = {strat_name: {} for strat_name, _, _ in strategy_configs}
    for strat_name, _, _ in strategy_configs:
        for m in range(0, M+2):
            avg_tp = np.mean(roc_data[strat_name][m]["TP"])
            avg_tn = np.mean(roc_data[strat_name][m]["TN"])
            avg_fp = np.mean(roc_data[strat_name][m]["FP"])
            avg_fn = np.mean(roc_data[strat_name][m]["FN"])
            # print(m, avg_tp, avg_tn, avg_fp, avg_fn)
            avg_roc[strat_name][m] = (avg_tp, avg_tn, avg_fp, avg_fn)

    # Save the computed avg_roc for the current M so it's not overwritten.
    all_avg_roc[M] = avg_roc

    # =============================================================================
    # 7. Plot ROC-like Curves for the current M on the subplot axes
    # =============================================================================

    marker_dict = {"Greedy": "o", "High Temp + Min-P": "s", "High Temperature": "D", "Top-k": "x", "Self-Consistency": "^"}
    color_dict = {"Greedy": "blue", "High Temp + Min-P": "purple", "High Temperature": "red", "Top-k": "green", "Self-Consistency": "orange"}
    legend_handles = {}
    # Second row: Plot Average TP vs Average FP.
    for strat_name in avg_roc:
        strat_params = extract_params(strat_name)
        if strat_params['temperature'] is None:
            if 'ensemble' in strat_name:
                label = 'Self-Consistency'
            else:
                label = 'Greedy'
        elif strat_params['min_p'] is not None and strat_params['min_p'] > 0.0:
            label = "High Temp + Min-P"
        elif strat_params['temperature'] >= 1.0:
            label = "High Temperature"
        else:
            # print(strat_params)
            label = "Top-k"
        if strat_params['top_k'] == 0:
            print("Messed up", strat_name)
            continue
        thresholds = sorted(avg_roc[strat_name].keys())
        tp_vals = [avg_roc[strat_name][m][0] / len(fp_model_names) for m in thresholds]
        fp_vals = [avg_roc[strat_name][m][2] / len(non_fp_model_names) for m in thresholds]
        # if label == 'Nucleus Sampling':
        #     print(strat_name, tp_vals, fp_vals, strat_params)
        # tp_vals = [roc_data[strat_name][m]["TP"] / len(fp_model_names) for m in thresholds]
        # fp_vals = [roc_data[strat_name][m]["FP"] / len(non_fp_model_names) for m in thresholds]
        if M == 1: alpha = 0.5
        # elif M == 5: alpha = 0.5
        elif M >= 10: alpha = 0.1
        else: alpha = 0.3
        sc = plt.scatter(fp_vals, tp_vals, marker=marker_dict[label], color=color_dict[label], alpha=alpha)
        (sp,) = plt.plot(fp_vals, tp_vals, color=color_dict[label], alpha=0.3)
        if label not in legend_handles:
            legend_handles[label] = sp
    # Plot y=x line for reference.
    plt.plot([0, 1], [0, 1], color='grey', linestyle='--')
    plt.xlabel("Average False Positives")
    plt.ylabel("Average True Positives")
    plt.title("M = {}".format(M))
    plt.grid(True)
    # Show single legend
    if i == 0:
        plt.legend(legend_handles.values(), legend_handles.keys(), loc="lower right")

    plt.tight_layout()
    plt.show()
    plt.close()

# Optionally, you can later access all_avg_roc to inspect the results for different M values.


Messed up temp_1.0-p_1.0-k_0
Messed up temp_1.0-p_1.0-k_0
Messed up temp_1.0-p_1.0-k_0
Messed up temp_1.0-p_1.0-k_0
